In [4]:
import os
import shutil
from PIL import Image

def process_images(source_folder):
    # Define paths for new folders
    processed_folder = os.path.join(source_folder, 'processed_images')
    rejected_folder = os.path.join(source_folder, 'rejected_images')

    # Create folders if they don't exist
    os.makedirs(processed_folder, exist_ok=True)
    os.makedirs(rejected_folder, exist_ok=True)

    # Supported image extensions
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

    # Loop through files in the source folder
    for filename in os.listdir(source_folder):
        if filename.lower().endswith(valid_extensions):
            file_path = os.path.join(source_folder, filename)
            
            try:
                with Image.open(file_path) as img:
                    width, height = img.size
                    
                    # Check if image is larger than 800x600
                    if width > 799 and height > 599:
                        # Downsize to 800x600
                        # Note: This forces the exact dimensions, potentially changing aspect ratio.
                        # To keep aspect ratio, use img.thumbnail((800, 600)) instead.
                        resized_img = img.resize((800, 600))
                        
                        # Save to processed folder
                        save_path = os.path.join(processed_folder, filename)
                        resized_img.save(save_path)
                        print(f"Processed: {filename} ({width}x{height} -> 800x600)")
                    else:
                        # Reject image
                        # Copy the original file to the rejected folder
                        shutil.copy2(file_path, rejected_folder)
                        print(f"Rejected: {filename} ({width}x{height} is not > 800x600)")
                        
            except Exception as e:
                print(f"Error processing {filename}: {e}")

# --- Usage ---
# You can change this path to point to your specific folder if needed
# Since you uploaded 'TestImage.rar', you would extract it and use that folder path.
source_directory = 'MultipleBalls' 

# Check if the directory exists before running (useful if running locally)
if os.path.exists(source_directory):
    process_images(source_directory)
else:
    print(f"Directory '{source_directory}' not found. Please extract your RAR file first.")

Rejected: $_12.jpeg (500x341 is not > 800x600)
Processed: $_57.jpeg (1480x1052 -> 800x600)
Rejected: -473Wx593H-4915680380-multi-MODEL (1).jpg (100x75 is not > 800x600)
Rejected: 0152b645ba06768718e8c6fd8a5f0d17.jpg (735x490 is not > 800x600)
Rejected: 120-140-standard-club-6-12-krg-sgb-cricket-leather-ball-sg-original-imaevs3q9xp6xezx.jpeg (480x426 is not > 800x600)
Rejected: 1711562536392_3388310092262510399578712428094777135967826n.jpg (540x960 is not > 800x600)
Rejected: 4-Swing (1).jpg (100x76 is not > 800x600)
Processed: 61zu4Ek06bL.jpg_BO30,255,255,255_UF900,850_SR1910,1000,0,C_QL100_.jpg (1910x1000 -> 800x600)
Processed: 66267d58f33f0c48743a34c4-kd-cricket-leather-ball-genuine-hand.jpg (1500x673 -> 800x600)
Rejected: 6925dc6d16ebc05566e741f2429b3cf4.jpg (363x249 is not > 800x600)
Rejected: 71C7bs9HVLL.jpg (640x1163 is not > 800x600)
Processed: 71oW-Fp7WUL._AC_UF894,1000_QL80_.jpg (894x927 -> 800x600)
Processed: 72108160.jpg (2450x1837 -> 800x600)
Rejected: 80-cricket-white-leat

In [3]:
import os
from PIL import Image, ImageDraw, ImageFont

def create_numbered_mosaic_grid(source_folder, output_folder, rows=8, cols=8, gap=10, bg_color='white'):
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get all valid image files
    files = [f for f in os.listdir(source_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]

    if not files:
        print(f"No images found in '{source_folder}'.")
        return

    # Function to try and load a decent font size
    def load_font(size=30):
        font_names = ["arial.ttf", "DejaVuSans.ttf", "FreeSans.ttf", "tahoma.ttf"]
        for name in font_names:
            try:
                return ImageFont.truetype(name, size)
            except IOError:
                continue
        # Fallback if no custom font found
        print("Warning: Could not load custom font, using default (might be small).")
        return ImageFont.load_default()

    # Load font once
    font = load_font(size=30)

    print(f"Found {len(files)} images. Creating numbered mosaic grids...")

    for filename in files:
        file_path = os.path.join(source_folder, filename)
        
        try:
            with Image.open(file_path) as img:
                if img.mode != 'RGB':
                    img = img.convert('RGB')

                width, height = img.size
                
                # Calculate tile sizes
                cell_w = width // cols
                cell_h = height // rows

                # Calculate canvas size
                new_width = (cell_w * cols) + (gap * (cols - 1))
                new_height = (cell_h * rows) + (gap * (rows - 1))

                # Create blank canvas
                grid_img = Image.new('RGB', (new_width, new_height), bg_color)
                draw = ImageDraw.Draw(grid_img)

                # Loop through rows and cols
                for row in range(rows):
                    for col in range(cols):
                        # 1. Crop Tile
                        left = col * cell_w
                        upper = row * cell_h
                        right = left + cell_w
                        lower = upper + cell_h
                        
                        tile = img.crop((left, upper, right, lower))

                        # 2. Paste Tile
                        paste_x = col * (cell_w + gap)
                        paste_y = row * (cell_h + gap)
                        grid_img.paste(tile, (paste_x, paste_y))

                        # 3. Add Number (1 to 64)
                        cell_num = row * cols + col + 1
                        text = str(cell_num)
                        
                        # Position text in top-left of the cell (with small padding)
                        text_x = paste_x + 5
                        text_y = paste_y + 5

                        # Draw text: Red color with White outline for visibility
                        draw.text(
                            (text_x, text_y), 
                            text, 
                            fill="red", 
                            font=font, 
                            stroke_width=2, 
                            stroke_fill="white"
                        )

                # Save result
                save_path = os.path.join(output_folder, filename)
                grid_img.save(save_path)
                print(f"Saved numbered mosaic: {filename}")

        except Exception as e:
            print(f"Failed to process {filename}: {e}")

    print(f"\nDone! Numbered images saved to '{output_folder}'")

# --- Configuration ---
input_dir = 'Raw/processed_images' 
output_dir = 'Raw/processed_images_with_grid'

# Settings
gap_size = 5       
background = 'white'

# Run
if os.path.exists(input_dir):
    create_numbered_mosaic_grid(input_dir, output_dir, gap=gap_size, bg_color=background)
else:
    print(f"Input folder '{input_dir}' does not exist.")

Found 96 images. Creating numbered mosaic grids...
Saved numbered mosaic: 162f10cf-ca1c-437a-ae8b-f572ff33ae6b.jpeg
Saved numbered mosaic: 408374.jpg
Saved numbered mosaic: 408377.jpg
Saved numbered mosaic: 408379.jpg
Saved numbered mosaic: 408381.jpg
Saved numbered mosaic: 408383.jpg
Saved numbered mosaic: 408384.jpg
Saved numbered mosaic: 5749f0ba-5d77-4a2a-bc40-568d3d214c53.jpg
Saved numbered mosaic: 5b47f5e5-1ede-45eb-854c-bf07e6f4006f.jpeg
Saved numbered mosaic: 9118ad62-0839-4de0-af76-afeb7a8fbc62.jpeg
Saved numbered mosaic: Australia_vs_India.jpg
Saved numbered mosaic: bb3c9281-58cb-4376-8d85-5c9b587310bf.jpeg
Saved numbered mosaic: Copy of Flick-1024x889.jpeg
Saved numbered mosaic: d776ac3d-0ba3-47be-84dc-24af35be3c4c.jpeg
Saved numbered mosaic: fde3e19b-9ebd-4235-b9ff-ec3ac3325a1d.jpg
Saved numbered mosaic: Flick-1024x889.jpeg
Saved numbered mosaic: india - eng1.png
Saved numbered mosaic: pexels-arsal-point-356971417-31131697.jpg
Saved numbered mosaic: pexels-arsal-point-35697

In [5]:
import os

def rename_images(folder_name):
    # Get the current working directory
    current_path = os.getcwd()
    
    # Construct the full path to the dataset folder
    folder_path = os.path.join(current_path, folder_name)

    # Check if the folder exists
    if not os.path.exists(folder_path):
        print(f"Error: The folder '{folder_name}' was not found.")
        return

    # Get a list of files and sort them (to ensure order isn't random)
    files = sorted(os.listdir(folder_path))
    
    # Counter for the numbering
    count = 1

    print(f"Starting rename process in: {folder_path}...\n")

    for filename in files:
        # Check if the file is an image (add/remove extensions as needed)
        # We use .lower() to handle .JPG vs .jpg
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')):
            
            # Extract the original file extension
            _, extension = os.path.splitext(filename)
            
            # Create the new file name
            new_name = f"Image{count}{extension}"
            
            # Define the full old and new paths
            old_file_path = os.path.join(folder_path, filename)
            new_file_path = os.path.join(folder_path, new_name)
            
            # Check if the new name already exists to prevent overwriting
            if not os.path.exists(new_file_path):
                os.rename(old_file_path, new_file_path)
                print(f"Renamed: {filename} -> {new_name}")
            else:
                print(f"Skipped: {new_name} already exists.")
            
            count += 1

    print("\nProcess Completed.")

# Run the function
rename_images("extra")

Starting rename process in: C:\Users\Student\IITB\Sem1part2\Project\extra...

Renamed: $_57.jpeg -> Image490.jpeg
Renamed: 61zu4Ek06bL.jpg_BO30,255,255,255_UF900,850_SR1910,1000,0,C_QL100_.jpg -> Image491.jpg
Renamed: 66267d58f33f0c48743a34c4-kd-cricket-leather-ball-genuine-hand.jpg -> Image492.jpg
Renamed: 71oW-Fp7WUL._AC_UF894,1000_QL80_.jpg -> Image493.jpg
Renamed: 72108160.jpg -> Image494.jpg
Renamed: 817gYHT3BWL._AC_UF894,1000_QL80_.jpg -> Image495.jpg
Renamed: 81aAfRBz-+L._AC_UF894,1000_QL80_.jpg -> Image496.jpg
Renamed: 81peX5AloFL.jpg -> Image497.jpg
Renamed: 81vwydSicjL.jpg_BO30,255,255,255_UF900,850_SR1910,1000,0,C_QL100_.jpg -> Image498.jpg
Renamed: 91EzCU9QkBL._AC_UF894,1000_QL80_.jpg -> Image499.jpg
Renamed: Image347.jpg -> Image500.jpg
Renamed: Image358.jpg -> Image501.jpg
Renamed: Image375.png -> Image502.png
Renamed: Image391.jpg -> Image503.jpg
Renamed: Screenshot_20251209-233544.png -> Image504.png
Renamed: Screenshot_20251209-233735.png -> Image505.png
Renamed: black